# From a Transformer Backbone to an Large Language Model

## The Generative Objective: Causal Language Modeling

In the previous lecture, the embedding model was trained using Masked Language Modeling, functioning as a bidirectional classifier to predict a hidden token based on surrounding context. Generative Large Language Models (LLMs) shift this objective to Causal Language Modeling. Instead of predicting a masked word in the middle of a sentence, the model performs a multi-class classification task to predict the probability distribution of the next token, given strictly the preceding sequence: $P(w_t|w_1,w_2,...,w_{t-1})$. To prevent the self-attention mechanism from accessing future tokens during training, a causal mask is applied to the attention weights. This mask zeroes out the upper triangle of the attention matrix, forcing each token to only attend to itself and the tokens that came before it.

## Decoding Strategies

The final layer of a language model outputs logits that are passed through a softmax function to create a probability distribution over the entire vocabulary. Because students are already familiar with classification networks, the computational challenge in text generation focuses on selecting the sequence of tokens from these step-by-step distributions. **Greedy search** selects the token with the highest probability at each step. While highly efficient, this deterministic approach frequently leads to repetitive and unnatural text loops. **Beam search** tracks the most probable sequence paths simultaneously, improving the overall likelihood of the generated sequence, but it remains largely deterministic. To achieve natural, human-like generation, stochastic sampling methods are utilized. **Temperature scaling** divides the logits by a constant prior to the softmax operation, where a value less than 1 sharpens the distribution and a value greater than 1 increases randomness. **Top-p (Nucleus) sampling** restricts the selection pool to the smallest set of tokens whose cumulative probability exceeds a specified threshold.

## Post-Training: Alignment and Instruction Tuning

A base model trained solely on causal language modeling will accurately estimate text distributions, but it will not inherently follow instructions or format its output as a helpful dialogue. Transforming this hierarchical feature extractor into a functional conversational agent requires post-training alignment. **Supervised Fine-Tuning (SFT)** updates the model weights using high-quality, human-annotated prompt-response pairs. This teaches the model the structural format of interaction, such as answering questions rather than simply appending text to them. To further align the model with human preferences, methods like **Reinforcement Learning from Human Feedback (RLHF)** or **Direct Preference Optimization (DPO)** are applied. These techniques encourage the LLM to generate "preferred" text (note the distinction from "accurate" or "helpful" or "safe").

## Interacting with the Tool: Prompt Templating and Context

A next-token predictor fundamentally continues a sequence. To function as a question-answering tool, raw user input is not passed to the model directly. Instead, the application layer wraps the input in a structured template using special control tokens. This template formats the interaction as a standardized dialogue script. For example, the user input "what is the weather?" is concatenated into a formatted string such as:

`<|system|>\nYou are a helpful assistant.\n<|user|>\nWhat is the weather?\n<|assistant|>\n`

The model receives this entire structured sequence, computes the probability distribution for the next token, and begins generation. Because it was aligned during SFT to complete sequences ending in the `<|assistant|>` token, the highest probability next tokens will form the answer. Generation continues iteratively, appending each new token to the input sequence, until the model outputs a designated stop token.

When utilizing this aligned tool, the user must operate within the model's **context window**. Because the transformer architecture processes the input sequence simultaneously through stacked multi-head self-attention mechanisms, the model lacks persistent state memory. In a multi-turn chat application, the entire preceding dialogue history, formatted with control tokens, must be appended to the new prompt, processed via sub-word tokenization, and fed back through the network to generate the next response. **In-context learning** relies entirely on this mechanism. Providing explicit examples, rules, or system instructions within the prompt temporarily conditions the output probability distribution of the sequence, allowing the model to adapt to new tasks without any updates to its underlying weights.

## Factuality and Hallucination Mitigation

Because LLMs are probabilistic sequence generators, they lack an inherent understanding of objective truth. This mathematical framework can result in **hallucinations**: the generation of fluent and highly plausible, yet factually incorrect, output. Mitigating hallucinations requires engineering the model's inputs and outputs to constrain its probability distributions toward factual accuracy.

* **Decoding Parameter Adjustment:** Lowering the temperature parameter reduces the probability mass assigned to unlikely tokens. This shifts the model toward deterministic output, decreasing the chance of generating "creative" but inaccurate facts.
* **System Prompting and Constraints:** Injecting explicit instructions into the `<|system|>` prompt (e.g., "Answer only using the provided text. If the answer is not present, output 'I do not know'") alters the conditional probabilities of the sequence, heavily penalizing fabricated responses.
* **Retrieval-Augmented Generation (RAG):** The parametric memory (internal weights) of a model is static and prone to recall errors. RAG architectures address this by querying an external, verified database for relevant documents based on the user's input. These retrieved documents are dynamically inserted into the model's context window before generation. This mechanism forces the self-attention layers to attend directly to factual, external text, grounding the generated response and significantly reducing hallucination rates.

## LLM Demo

Here we demonstrate with a small (and less capable) LLM how we can interact with one to build fluent responses to prompts. First we download the model.

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import os
import httpx
from huggingface_hub import constants

# 1. Standard environment bypasses
os.environ['HF_HUB_DISABLE_SSL_VERIFY'] = '1'
os.environ['CURL_CA_BUNDLE'] = ''

# 2. Force httpx to ignore SSL structural errors (Missing Authority Key Identifier)
# We wrap the original client to ensure verify=False is always passed.
original_client = httpx.Client

class UnverifiedClient(original_client):
    def __init__(self, *args, **kwargs):
        kwargs['verify'] = False
        super().__init__(*args, **kwargs)

httpx.Client = UnverifiedClient

# 3. Suppress warnings
import warnings
from urllib3.exceptions import InsecureRequestWarning
import requests
requests.packages.urllib3.disable_warnings(category=InsecureRequestWarning)
warnings.filterwarnings('ignore')

# Define the lightweight, pre-trained instruction-tuned backbone
model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

# Ensure the model operates in evaluation mode
model.eval()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

You can see by the final layer that tere are 151,936 output nodes, meaning there are 151,936 different tokens this model can choose between.

This next code makes a function that can produce an engineered prompt from the user's input.

In [2]:
def format_prompt(user_query, system_instruction, context=None):
    """
    Wraps inputs into a structured conversational template.
    Injects context if provided.
    """
    if context:
        user_query = f"Context: {context}\n\nQuestion: {user_query}"
        
    # The standard template uses specific control tokens to denote roles
    template = (
        f"<|im_start|>system\n{system_instruction}<|im_end|>\n"
        f"<|im_start|>user\n{user_query}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    return template

# Test the templating function
system_msg = "You are a helpful assistant."
query = "What is the capital of France?"
prompt = format_prompt(query, system_msg)
print(prompt)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant



Here, the **system instructions** are "You are a precise, helpful assistant." Other system instructions might be, for instance, the following text which are the instructions to the Gemini Gem you're using on your facial recognition lab (there are also other instructions from Google in each prompt that we don't have visibility on to help define their product's behavior):

```
Your job is to be used by students who are trying to complete the attached assignment. Their responsibilities are described in the section labeled "Your Task". Different coding portions are labeled as "AI Yellow" or "AI Green".

Purpose and Goals:

* Assist students with the attached assignment while strictly adhering to the specified Generative AI policy levels: GREEN, YELLOW, and RED.
* Ensure students understand the underlying concepts by providing detailed explanations for permitted assistance.
* Maintain academic integrity by refusing requests that violate the designated color-coded restrictions.
* Once a color-coded label appears, the assignment remains under that label until it is labeled something else.

Behaviors and Rules:

1) Policy Enforcement:

 a) For sections labeled GREEN: Provide full assistance, including code and logic. Always include a comprehensive explanation so the student understands the output.
 b) For sections labeled YELLOW: Assist only with debugging or coding individual lines. Refuse requests for full functions or high-level logic/strategy. If a request is too broad, respond with: 'I believe that is a question on a yellow portion, and your request is too expansive.'
 c) For sections labeled RED: Refuse all assistance. Respond briefly with: 'I believe that is a question on a red portion, and I will not assist.'

2) Context and Notebook Analysis:

 a) Use the provided Jupyter notebook to determine the color policy for specific questions based on the HTML div tags (GREEN, YELLOW, RED).
 b) Assume all user queries relate to this specific assignment.
 c) Require students to provide context for their questions. Do not respond to vague requests like 'fill in the first cell' without the student providing specific content or a problem description.

3) Communication Style:

 a) Maintain a concise and precise tone.
 b) Avoid being chatty or overly familiar.
 c) Focus strictly on the academic content and policy compliance.

Overall Tone:

* Professional, academic, and strictly objective.
* Disciplined in following the established rules of the assignment policy.
```

The **query** here is the question our user has typed into the chatbot: "What is the capital of France?"

These sections are set apart with `<|im_start|>` and `<|im_end|>` tags. Finally, it starts the response from the assistant, but we're left dangling - it's time for the LLM to start recommending next tokens to complete this string!

Our next function chooses the most likely next token, and assembles the response - it will do so, until it outputs the token "end-of-sequence," which means it is done.

In [3]:
def manual_greedy_decode(prompt_text, max_new_tokens=100):
    """
    An explicitly coded autoregressive loop using greedy selection (argmax).
    """
    # Convert text to tensor of token IDs
    input_ids = tokenizer(prompt_text, return_tensors="pt").input_ids
    
    print("Generating:", end=" ", flush=True)
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Pass the sequence through the transformer
            outputs = model(input_ids)
            
            # Extract logits for the final token position in the sequence
            next_token_logits = outputs.logits[:, -1, :]
            
            # Greedy search: select the token ID with the highest probability
            next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)
            
            # Append the new token to the input sequence for the next iteration
            input_ids = torch.cat([input_ids, next_token_id], dim=-1)
            
            # Decode and print the single new token
            new_word = tokenizer.decode(next_token_id[0])
            print(new_word, end="", flush=True)
            
            # Terminate if the model outputs the End-Of-Sequence control token
            if next_token_id.item() == tokenizer.eos_token_id:
                break
                
    print("\n")
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

# Execute the manual loop
_ = manual_greedy_decode(prompt)

Generating: The capital of France is Paris.<|im_end|>



You get more interesting responses if you increase the "temperature," and allow for more randomness in output:

In [8]:
def generate_response(prompt_text, temperature=1.0, top_p=1.0, do_sample=False):
    """
    Utilizes the optimized generate method to test decoding parameters.
    """
    inputs = tokenizer(prompt_text, return_tensors="pt")
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=temperature,
            top_p=top_p,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id
        )
        
    # Isolate the generated response from the input prompt
    input_length = inputs.input_ids.shape[1]
    response_ids = output_ids[0][input_length:]
    return tokenizer.decode(response_ids, skip_special_tokens=True)

# Test 1: Deterministic (Greedy)
print("Deterministic Output:")
print(generate_response(prompt, do_sample=False))

# Test 2: Stochastic (High Temperature)
print("\nStochastic Output (T=1.5):")
print(generate_response(prompt, temperature=1.5, do_sample=True))

Deterministic Output:
The capital of France is Paris.

Stochastic Output (T=1.5):
The capital of France is Paris.

Historic facts and other details include:

1. Founded by Duke Charles I in 986 to protect the rights of Normans from Norman conquerors.
2. Originally part of both Îtalo-Cro


Of course, hallucinations are quite possible. Here's an example.

In [5]:
# 1. Induce a hallucination
fictional_query = "Who won the 1815 Super Bowl?"
hallucination_prompt = format_prompt(fictional_query, system_msg)

print("Base Model Response (Likely Hallucination):")
print(generate_response(hallucination_prompt, do_sample=False))

Base Model Response (Likely Hallucination):
The 1815 Super Bowl was won by the New York Giants, led by quarterback Eli Manning. The game was played at the Ford Field in New York City and featured a score of 27-10 in favor of the Giants


To mitigate this, we can add knowledge to the prompt.

In [6]:
# 2. Mitigate via Context Injection (RAG simulation)
factual_context = "The Super Bowl was first played in 1966, after the merger of the NFL and the AFL."
grounded_prompt = format_prompt(fictional_query, system_msg, context=factual_context)

print(grounded_prompt)

print("Context-Grounded Response (RAG):")
print(generate_response(grounded_prompt, do_sample=False))

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Context: The Super Bowl was first played in 1966, after the merger of the NFL and the AFL.

Question: Who won the 1815 Super Bowl?<|im_end|>
<|im_start|>assistant

Context-Grounded Response (RAG):
The 1815 Super Bowl was not held. It is possible that you may have made a mistake or there might be an error in your question. Could you please provide more information so I can assist you better?
